<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

JMG This notebook is adapted from Raschka's chapter 6 notebook.  The primary changes are

1. A new harder dataset for classifying (the Pang/Lee movie reviews dataset, available as a HuggingFace dataset).
2. Downloading a version of the Open AI GPT weights that is appropriate for a **large** GPT model.
3. Implementing saving the best model during training.
4. Implementing cross-validation during training.

# Chapter 6: Finetuning for Text Classification

In [1]:
from importlib.metadata import version

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.10.0
numpy version: 2.4.2
tiktoken version: 0.12.0
torch version: 2.5.1
tensorflow version: 2.21.0
pandas version: 2.2.3


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/01.webp" width=500px>

&nbsp;
### 6.1 Different categories of finetuning

- No code in this section

- The most common ways to finetune language models are instruction-finetuning and classification finetuning
- Instruction-finetuning, depicted below, is the topic of the next chapter

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/02.webp" width=500px>

- Classification finetuning, the topic of this chapter, is a procedure you may already be familiar with if you have a background in machine learning -- it's similar to training a convolutional network to classify handwritten digits, for example
- In classification finetuning, we have a specific number of class labels (for example, "spam" and "not spam") that the model can output
- A classification finetuned model can only predict classes it has seen during training (for example, "spam" or "not spam"), whereas an instruction-finetuned model can usually perform many tasks
- We can think of a classification-finetuned model as a very specialized model; in practice, it is much easier to create a specialized model than a generalist model that performs well on many different tasks

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/03.webp" width=400px>

&nbsp;
### 6.2 Preparing the dataset

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/04.webp" width=500px>

## Another dataset (Movie reviews)

In [ ]:
#from datasets import load_dataset_builder,get_dataset_split_names
#from datasets import load_dataset
#dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train")
# All 3 splits
#dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

In [4]:
from datasets import load_dataset
from datasets import load_from_disk
import os.path
import os

HF_TOKEN = os.environ["HF_TOKEN"]

data_dir = "/Users/gawron/Desktop/src/sphinx/"\
"python_for_ss_extras/colab_notebooks/transformers/HuggingFaceTransformers/"\
"raschke/LLMs-from-scratch/"
dataset_nm = "cornell-movie-review"
dataset_nm = f"{dataset_nm}-all"
dataset_huggingface_repo = "cornell-movie-review-data/rotten_tomatoes"

#test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test",token=HF_TOKEN)
if os.path.isdir(os.path.join(data_dir,dataset_nm)):
    mr_data = load_from_disk(dataset_nm)
else:
    mr_data = load_dataset(dataset_huggingface_repo,token=HF_TOKEN)
    # save local copy for next time
    mr_data.save_to_disk(dataset_nm)

Saving the dataset (0/1 shards):   0%|          | 0/8530 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1066 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1066 [00:00<?, ? examples/s]

In [8]:
mr_train = mr_data["train"]
mr_val = mr_data["validation"]
mr_test = mr_data["test"]

In [9]:
#from chapter02 import create_dataloader_v1
from torch.utils.data import Dataset, DataLoader
from datasets.arrow_dataset import Dataset as ArrowDataset
import time
import pandas as pd

max_length,stride = 256, 128

class MovieReviewDataset(Dataset):
    #class SpamDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=None, pad_token_id=50256):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncate sequences if they are longer than max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data[index]["label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
        # Note: A more pythonic version to implement this method
        # is the following, which is also used in the next chapter:
        # return max(len(encoded_text) for encoded_text in self.encoded_texts)    

def get_cross_validation_splits(train,val,test,nfolds=3,seed=47,
                               ):
    """
    train,val,test are Dataset instances (e.g., datasets.arrow_dataset.Dataset).
    
    Merge data, split into nfolds pieces, create train,val,test loaders
    for each fold.  
    
    this is a generator.
    """
    items = (train, val, test)
    df = pd.concat([item.to_pandas() for item in items])
    #Idiom for shuffling data
    df = df.sample(frac=1,random_state=seed).reset_index(drop=True)
    fold_len = len(df)//nfolds
    dfs = [df.iloc[i*fold_len:(i*fold_len)+fold_len] for i in range(nfolds)]
    #len of val,test sets assuming equal size
    split_len = len(dfs[0])//2
    for i in range(nfolds):
        val_test = dfs[i]
        val, test = [val_test.iloc[(i*split_len):((i*split_len)+split_len)] for i in range(2)]
        val, test = ArrowDataset.from_pandas(val),ArrowDataset.from_pandas(test)
        train = pd.concat(dfs[:i] + dfs[i+1:])
        train = ArrowDataset.from_pandas(train)
        yield (train, val, test)
        
def run_mr_cross_vals(mr_train, mr_val, mr_test, model_file=None, nfolds=3,seed=123,
                      batch_size=20, num_workers=0,num_epochs=8,
                     lr=5e-5,eval_freq=50,eval_iter=5,
                     save_path="model.pt"):
    """
    If model file is not None must pass in full path.
    """
    global results,mr_test_loader
    results= []
    for (i,(train, val, test)) in enumerate(
                                        get_cross_validation_splits(mr_train,mr_val,
                                                          mr_test,nfolds=nfolds,
                                                          seed=seed)
    ):

        mr_train_loader = DataLoader(
            dataset=MovieReviewDataset(train,tokenizer),
            batch_size=batch_size,
            drop_last=True,
            shuffle=True,
            num_workers=num_workers
        )
        mr_val_loader = DataLoader(
            MovieReviewDataset(val,tokenizer),
            batch_size=batch_size,
            drop_last=False,
            shuffle=False,
            num_workers=num_workers
        )
        mr_test_loader = DataLoader(
            MovieReviewDataset(test,tokenizer),
            batch_size=batch_size,
            drop_last=False,
            shuffle=False,
            num_workers=num_workers
        )
        result = run_training (mr_train_loader,  mr_val_loader, 
                               model_file=model_file, num_epochs=num_epochs, 
                               lr=lr, eval_freq=eval_freq, eval_iter=eval_iter,
                               save_path=save_path,seed=seed)
        
        results.append(result)
        #test_accuracy0 = calc_accuracy_loader(data_loader, model, device)
        print(f"***Getting test results for validation run {i+1}!***")
        # fully trained model is the denotation of global variable `model`
        #OR eval_model=model,save_path=None
        result["test_acc"] = get_test_results(mr_test_loader,num_epochs,
                                              save_path=save_path,seed=seed)
        print("***End Testing!***")
    avg_acc = sum([result["test_acc"] for result in results])/nfolds
    print(f"Average accuracy over {nfolds}: {avg_acc:.2f}")
    return avg_acc, results
    
def run_training (train_loader, val_loader, model_file=None, num_epochs=8, lr=5e-5, 
                   eval_freq=50,eval_iter=5,save_path="model.pt",seed=123):
    """
    Creates model and runs num_epochs of training.  
    
    Wrapper for train_classifier_simple.
    
    Alseo creates optimizer and a few other housekeeping bits
    and calls train_classifier_simple.
    
    If model_file is None, fetches the large GPT model as default.
    
    If model file is not None, must pass in full path, for example 
    the global wt_file_name, which loads the small GPT model.
    
    global device must also be set.
    
    Returns result dictionary.
    """
    global model
    start_time = time.time()
    torch.manual_seed(seed)
    #  Clear out some memory for a BIG object
    if "model" in globals().keys():
        del model
    model= make_classifier_model(path=model_file)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    train_losses, val_losses, train_accs, val_accs, examples_seen =\
      train_classifier_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=num_epochs, eval_freq=eval_freq, eval_iter=eval_iter,
        save_path=save_path
        )
    end_time = time.time()
    execution_time_minutes = (end_time - start_time) / 60
    print(f"{num_epochs} epochs of training completed with {lr=}"\
          f" in {execution_time_minutes:.2f} minutes.")
    result = dict(train_losses=train_losses,
                 val_losses=val_losses,
                 train_accs=train_accs,
                 val_accs=val_accs)
    return result

def get_test_results(data_loader, n_epochs, eval_model=None, save_path=None,
                     seed=None, clobber=True):
    global model
    print("save_path",save_path)
    if save_path is not None:
        #  Clear out some memory for a BIG object model var shd be defined
        #  if not, this will raise an error
        if clobber and "model" in globals().keys():
            print(f"Test: Clobbering epoch {n_epochs} model.")
            del model
        model = load_saved_classifier_model (save_path)
        print(f"Using best model.")
    else:
        model = eval_model
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    with torch.no_grad():
        #test_loss = calc_loss_loader(data_loader, model, device)
        test_accuracy = calc_accuracy_loader(data_loader, model, device)
    print(f"Test accuracy:  {test_accuracy:.2%}")
    return test_accuracy

- This section prepares the dataset we use for classification finetuning
- We use a dataset consisting of spam and non-spam text messages to finetune the LLM to classify them
- First, we download and unzip the dataset

&nbsp;
### 6.3 Creating data loaders

- Note that the text messages have different lengths; if we want to combine multiple training examples in a batch, we have to either
  1. truncate all messages to the length of the shortest message in the dataset or batch
  2. pad all messages to the length of the longest message in the dataset or batch

- We choose option 2 and pad all messages to the longest message in the dataset
- For that, we use `<|endoftext|>` as a padding token, as discussed in chapter 2

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/06.webp" width=500px>

In [25]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


Data items are a pair of n encodings tensior and a class label.

In [11]:
mr_train[0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [13]:
train_dataset = MovieReviewDataset(mr_train,tokenizer)
train_dataset[14]

(tensor([  586,  3358,   837,   340, 16723,   364,   262,  3840,   356,   761,
          3923,   523,   881,   764, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]),
 tensor(1))

In [30]:
import numpy as np

def count_padding  (item):
    return  (item[0]==50256).sum()

a = np.array([count_padding(item) for item in train_dataset])
idx = np.argmax(-1*a).item()
idx

7684

Here is the longest review, which determines the length to which all the others are padded.

In [31]:
train_dataset[idx]

(tensor([   78,  6184,   111, 16514,    78,  1658,  1640, 16175,    78,   466,
         19958, 13165,   936, 15498,  3758,    78,  6730,    81,  4533, 16176,
            78,   686,   660,  7058,   837,  8358,   837,  1207, 10924,   390,
           443,  7785, 23781,  8626, 28691, 31215,   951,   420,   283,   257,
           491,  1689,   795,   290,  3263,    78,   837,   583,  2934,    12,
           325,   390,  1569,    89,   257,   636,   343,   466,  9113,    68,
           795,  8358, 28686,  1556,  2596,    71,   418,   936,   261, 36281,
          3681,   418,   264, 28749,  1193,   291, 22484,   764]),
 tensor(0))

Which, interestingly enough is in Portuguese:

In [33]:
mr_train[idx]

{'text': 'o ótimo esforço do diretor acaba sendo frustrado pelo roteiro , que , depois de levar um bom tempo para colocar a trama em andamento , perde-se de vez a partir do instante em que os estranhos acontecimentos são explicados .',
 'label': 0}

Which the tokenizer handles by splitting up the words into small pieces.

In [61]:
len(mr_train[idx]["text"].split()),len(train_dataset[idx][0])

(41, 78)

This is in fact a movie review, and a negative one.  The translation is

> The director's excellent effort is ultimately frustrated by the screenplay, which—after taking a good while to get the plot moving—completely loses its way the moment the strange events are explained.

- Next, we use the dataset to instantiate the data loaders, which is similar to creating the data loaders in previous chapters

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/07.webp" width=500px>

In [10]:
torch.manual_seed(123)
num_workers = 0
batch_size = 20

torch.manual_seed(123)

mr_train_loader = DataLoader(
    dataset=MovieReviewDataset(mr_train,tokenizer),
    batch_size=batch_size,
    drop_last=True,
    shuffle=True,
    num_workers=num_workers
)

mr_val_loader = DataLoader(
    MovieReviewDataset(mr_val,tokenizer),
    batch_size=batch_size,
    drop_last=False,
    shuffle=False,
    num_workers=num_workers
)

mr_test_loader = DataLoader(
    MovieReviewDataset(mr_test,tokenizer),
    batch_size=batch_size,
    drop_last=False,
    shuffle=False,
    num_workers=num_workers
)


&nbsp;
### 6.4 Initializing a model with pretrained weights

- In this section, we initialize the pretrained model we worked with in the previous chapter

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/08.webp" width=500px>

### Using a locally saved small model or Raschka's github site

The GPT model class defined in Chapter 5:

In [1]:
import torch
from torch import nn
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

class MultiHeadAttention(nn.Module):
    """
    x_out = self.forward(x_in)
    
    x_in: batch_size x context_length x d_in 
    
    x_out: batch_size x context_length x d_out
    
    Note that an instantiated learner is indifferent to 
    the particular value of batch_size. The assumption is
    simply that the input is a 3D matrix and the first dimension is batch
    size, but only context_length, d_in, and d_out are parameters of 
    the learner.
    
    The d_out dimensions will be evenly divided among the heads, so
    d_out must be evenly divisible by num_heads. 
    """
    def __init__(self, d_in, d_out, 
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # Will divide the d_out dimensions evenly among the heads
        self.head_dim = d_out // num_heads    #1
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)   
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, context_length, d_in = x.shape
        # keys is now b x context_length x d_out
        keys = self.W_key(x)         #3
        queries = self.W_query(x)    #3
        values = self.W_value(x)     #3

        # dividing d_out into num_heads components; each component has shape head_dim 
        # keys is now b x context_length x num_heads x head_dim
        keys = keys.view(b, context_length, self.num_heads, self.head_dim)       #4
        values = values.view(b, context_length, self.num_heads, self.head_dim)  
        queries = queries.view(                                             
            b, context_length, self.num_heads, self.head_dim                    
        )                                                                   


        # keys becomes b x num_heads x context_length x head_dim
        keys = keys.transpose(1, 2)          #5
        queries = queries.transpose(1, 2)    #5
        values = values.transpose(1, 2)      #5

        # make keys compatible for matrix multiplication 
        #   mapping     @  mapped
        #   (..., m, n) @ (..., n, m)
        # keys becomes b x num_heads x head_dim X context_length
        # attn_scores will be b x num_heads x context_length x context_length
        attn_scores = queries @ keys.transpose(2, 3)   #6
        mask_bool = self.mask.bool()[:context_length, :context_length]    #7

        attn_scores.masked_fill_(mask_bool, -torch.inf)     #8

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # attn_weights : b x num_heads x context_length x context_length
        # values :  b x num_heads x context_length x head_dim
        # context_vec:  b x num_heads x context_length x head_dim
        # transpose(1, 2) =>
        # context_vec:  b x context_length x num_heads x head_dim
        context_vec = (attn_weights @ values).transpose(1, 2)   #9
        # Recombine num_heads x head_dim into d_out
        # context_vec:  b x context_length x d_out
        context_vec = context_vec.contiguous().view(
            b, context_length, self.d_out
        )                                           #10
        # And one more linear map for good luck to produce what we might call logits;
        # 2D map is a square matrix:
        # context_vec:  b x context_length x d_out
        context_vec = self.out_proj(context_vec)    #11
        return context_vec       

class LayerNorm(nn.Module):
    """
    
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

# We wrap it in a feed forward layer
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)
    
class TransformerBlock(nn.Module):
    """
    Required __init__ argument is a GPT 
    configuration dictionary (GPT_CONFIG_124M in this NB).
    """
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x  #1
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #2

        shortcut = x         #3
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #4
        return x
#1 Shortcut connection for attention block
#2 Add the original input back
#3 Shortcut connection for feed forward block
#4 Adds the original input back

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
        
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
 #1
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits
    
    
#1 The device setting will allow us to train 
#  the model on a CPU or GPU, depending on which device the input data sits on.

#GPT_CONFIG_124M = {
#    "vocab_size": 50257,     # Vocabulary size.  Weve already used this vocab in teh GPOT tolkenzier
#    "context_length": 1024,  # Context length.  Hoo boy!  Bug step up
#    "emb_dim": 768,          # Embedding dimension    More than doubling the embedding dimensionality above
#    "n_heads": 12,           # Number of attention heads.  More memory
#    "n_layers": 12,          # Number of layers.  More meory still!
#    "drop_rate": 0.1,        # Dropout rate  modest
#    "qkv_bias": False        # Query-Key-Value bias. for later
#}

#gpt = GPT_CONFIG_124M

# NB This class is used exclusively as the template for loading
# a state dict into
class GPTModelCLF(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        #self.out_head = nn.Linear(
        #    cfg["emb_dim"], cfg["vocab_size"], bias=False
        #)
        self.out_head = nn.Linear(in_features=cfg["emb_dim"], 
                                  out_features=cfg["num_classes"],
                                  bias=False)
        
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
 #1
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits



[50256]


Define configurations

In [2]:
#CHOOSE_MODEL = "gpt2-small (124M)"
CHOOSE_MODEL = "gpt2-large (774M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True,         # Query-key-value bias
    "num_classes": 2         # Only for the classsifier
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

Create the model instance:

Instead of the code in the next cell you use the notebook weight-loading-pytorch
from the Ch 5 section 2 code (alternative weight loading) published on the [Building a Large Language Model from Scratch github site.](https://github.com/rasbt/LLMs-from-scratch)

In [3]:
try:
    from accelerate import Accelerator
    use_accelerator=True
except:
    print("accelerator unavailble")
    use_accelerator=False
    

import os.path

def initialize_GPT_model(file_name, config,use_accelerator=use_accelerator):
    """
    Assumption this works for bioth large and small models.
    
    Distinctions encoded in config. and file_name, which must agree on model size.
    """
    model = GPTModel(config)
    device= choose_device(use_accelerator=use_accelerator)
    # Load locally save weights
    model.load_state_dict(torch.load(file_name, weights_only=True))
    model.to(device)
    return model

def initialize_GPT_model_no_wts(config,use_accelerator=use_accelerator):
    model = GPTModel(config)
    device= choose_device(use_accelerator=use_accelerator)
    model.to(device)
    return model

def initialize_GPTCLF_model_no_wts(config,use_accelerator=use_accelerator):
    model = GPTModelCLF(config)
    device= choose_device(use_accelerator=use_accelerator)
    model.to(device)
    return model

def choose_device (use_accelerator=False):
    if use_accelerator:
        return Accelerator().device
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        # Use PyTorch 2.9 or newer for stable mps results
        major, minor = map(int, torch.__version__.split(".")[:2])
        if (major, minor) >= (2, 9):
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
    else:
        device = torch.device("cpu")
    return device

In [4]:
notebook_dir = '/Users/gawron/Desktop/src/sphinx/python_for_ss_extras/colab_notebooks/'
llms_from_scratch = 'transformers/HuggingFaceTransformers/raschke/LLMs-from-scratch/'
# this should end up being the directory you have cloned the LLM book github site into
llms_from_scratch = os.path.join(notebook_dir, llms_from_scratch)
weights_dir = 'ch05/02_alternative_weight_loading'
weights_dir  = os.path.join(llms_from_scratch, weights_dir)
wt_file_name = "gpt2-small-124M.pth"
wt_file_name = os.path.join(weights_dir,wt_file_name)
#model = GPTModel(BASE_CONFIG)
#model.load_state_dict(torch.load(file_name, weights_only=True))
#model.eval()

This uses a weight file downloaded locally in previous chapters.

model = initialize_GPT_model(wt_file_name, config=BASE_CONFIG,use_accelerator=use_accelerator)
model.eval()

### Using OpenAI models site or Raschka's copies (needs tensorflow)

The downloading code below needs tensorflow:

Currently this is the only way I know of to get a large model with the Ch. 5 architecture.

In [4]:
import os
import urllib.request
# import requests
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm

from tqdm import tqdm
import os.path

def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    backup_base_url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        backup_url = os.path.join(backup_base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        print(f"Downloading {file_url}/{file_path}")
        download_file(file_url, file_path, backup_url)
        print(f"Download complete")

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params


def download_file(url, destination, backup_url=None):
    def _attempt_download(download_url):
        with urllib.request.urlopen(download_url) as response:
            # Get the total file size from headers, defaulting to 0 if not present
            file_size = int(response.headers.get("Content-Length", 0))

            # Check if file exists and has the same size
            if os.path.exists(destination):
                file_size_local = os.path.getsize(destination)
                if file_size == file_size_local:
                    print(f"File already exists and is up-to-date: {destination}")
                    return True  # Indicate success without re-downloading

            block_size = 1024  # 1 Kilobyte

            # Initialize the progress bar with total file size
            progress_bar_description = os.path.basename(download_url)
            with tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
                with open(destination, "wb") as file:
                    while True:
                        chunk = response.read(block_size)
                        if not chunk:
                            break
                        file.write(chunk)
                        progress_bar.update(len(chunk))
            return True

    try:
        if _attempt_download(url):
            return
    except (urllib.error.HTTPError, urllib.error.URLError):
        if backup_url is not None:
            print(f"Primary URL ({url}) failed. Attempting backup URL: {backup_url}")
            try:
                if _attempt_download(backup_url):
                    return
            except urllib.error.HTTPError:
                pass

        # If we reach here, both attempts have failed
        error_message = (
            f"Failed to download from both primary URL ({url})"
            f"{' and backup URL (' + backup_url + ')' if backup_url else ''}."
            "\nCheck your internet connection or the file availability.\n"
            "For help, visit: https://github.com/rasbt/LLMs-from-scratch/discussions/273"
        )
        print(error_message)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        
def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])


In [5]:
import os.path

#CHOOSE_MODEL = "gpt2-small (124M)"
CHOOSE_MODEL = "gpt2-large (774M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
models_dir ="gpt2"
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
model_dir = os.path.join(models_dir, model_size)
path = os.path.join(model_dir,"model.pt")

In [38]:
settings, params = download_and_load_gpt2(model_size=model_size, models_dir=models_dir)
#model = GPTModel(BASE_CONFIG)

checkpoint: 100%|███████████████████████████| 77.0/77.0 [00:00<00:00, 8.28kiB/s]


Download complete


encoder.json: 100%|███████████████████████| 1.04M/1.04M [00:00<00:00, 1.84MiB/s]


Download complete


hparams.json: 100%|█████████████████████████| 91.0/91.0 [00:00<00:00, 23.8kiB/s]


...

Download complete


model.ckpt.meta: 100%|████████████████████| 1.38M/1.38M [00:00<00:00, 2.00MiB/s]


Download complete


vocab.bpe: 100%|████████████████████████████| 456k/456k [00:00<00:00, 1.17MiB/s]


Download complete


In [12]:
#tf_ckpt_path = os.path.join(model_dir, "model.ckpt.data-00000-of-00001")
tf_ckpt_path = os.path.join(model_dir, "model.ckpt")
settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

In [13]:
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

This is the basis of the model used for training below.

In [44]:
import os.path

def save_model (model, optimizer,epoch, loss, lr, path):
    torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
            'loss': loss,
            'lr':  lr
            }, path)

#models_dir,model_size="gpt2","124M"
model_dir = os.path.join(models_dir, model_size)
save_model(model,optimizer=None,epoch=0,loss=0,lr=5e-5,
           path=os.path.join(model_dir,"model.pt"))

## Loading the saved large model

In [38]:
#model = initialize_GPT_model(path, config=BASE_CONFIG,use_accelerator=use_accelerator)
#checkpoint = torch.load(path, weights_only=True)

def load_large_model ():
    """
    Loads large GPT model.
    
    Note: no classifier out_head.
    """
    models_dir ="gpt2"
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    model_dir = os.path.join(models_dir, model_size)
    path = os.path.join(model_dir,"model.pt")
    device = choose_device(use_accelerator=use_accelerator)
    model = initialize_GPT_model(path, config=BASE_CONFIG,use_accelerator=use_accelerator)
    #model.load_state_dict(torch.load(path, weights_only=True))
    model.to(device)
    model.eval()
    return model

In [39]:
model = load_large_model ()
model

GPTModel(
  (tok_emb): Embedding(50257, 1280)
  (pos_emb): Embedding(1024, 1280)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1280, out_features=1280, bias=True)
        (W_key): Linear(in_features=1280, out_features=1280, bias=True)
        (W_value): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1280, out_features=5120, bias=True)
          (1): GELU()
          (2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

##  Test what you loaded

- To ensure that the model was loaded correctly, let's double-check that it generates coherent text

In [40]:
assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)

In [6]:

#Listing 4.8 A function for the GPT model to generate text
# Note this function is redefined with the training code
def generate_text_simple(model, idx,                 #1
                         max_new_tokens, context_size): 
    for i in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]    #2
        #idx_cond.to(device)
        #model.to(device)
        with torch.no_grad():
            logits = model(idx_cond)
            try:
                logits = logits[:, -1, :]  
            except:
                logits = logits.logits[:, -1, :]  
        # batch x n_token x vocab_size) => batch x vocab_size
        #logits = logits[:, -1, :]                    #3
        # scores => weights, shape unchanged
        probas = torch.softmax(logits, dim=-1)           #4
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)    #5
        #print(i, idx_next.shape,idx.shape)
        idx = torch.cat((idx, idx_next), dim=1)     #6

    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove batch dimension
    return tokenizer.decode(flat.tolist())

In [15]:
# Alternatively:
# from llms_from_scratch.ch05 import (
#    generate_text_simple,
#    text_to_token_ids,
#    token_ids_to_text
# )


text_1 = "Every effort moves you"
device=choose_device(use_accelerator=use_accelerator)
model.to(device)
idx = text_to_token_ids(text_1, tokenizer).to(device)
idx = idx.reshape((1,-1))
print("Calling generate_text_simple")
token_ids = generate_text_simple(
    model=model,
    idx=idx,
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
token_ids = token_ids.reshape((1,-1))
print(token_ids_to_text(token_ids, tokenizer))

Calling generate_text_simple
Every effort moves you forward.

"I'm not going to be a guy who's


- Before we finetune the model as a classifier, let's see if the model can perhaps already classify spam messages via prompting

In [22]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
text_2 = (
    "Peter Piper picked a peck of pickled peppers"
)

idx = text_to_token_ids(text_2, tokenizer).to(device)
idx = idx.reshape((1,-1))
token_ids = generate_text_simple(
    model=model,
    idx=idx,
    max_new_tokens=42,
    context_size=BASE_CONFIG["context_length"]
)
token_ids = token_ids.reshape((1,-1))
print(token_ids_to_text(token_ids, tokenizer))

Peter Piper picked a peck of pickled peppers and a handful of fresh basil leaves to add to the mix.

"I'm not a big fan of garlic," he said. "I like to add a little bit of garlic to my food.


- As we can see, the model is not very good at following instructions
- This is expected, since it has only been pretrained and not instruction-finetuned (instruction finetuning will be covered in the next chapter)

&nbsp;
### 6.5 Adding a classification head

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/09.webp" width=500px>

- In this section, we are modifying the pretrained LLM to make it ready for classification finetuning
- Let's take a look at the model architecture first

In [54]:
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 1280)
  (pos_emb): Embedding(1024, 1280)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1280, out_features=1280, bias=True)
        (W_key): Linear(in_features=1280, out_features=1280, bias=True)
        (W_value): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1280, out_features=5120, bias=True)
          (1): GELU()
          (2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

- Above, we can see the architecture we implemented in chapter 4 neatly laid out
- The goal is to replace and finetune the output layer
- To achieve this, we first freeze the model, meaning that we make all layers non-trainable

In [48]:
for param in model.parameters():
    param.requires_grad = False

- Then, we replace the output layer (`model.out_head`), which originally maps the layer inputs to 50,257 dimensions (the size of the vocabulary)
- Since we finetune the model for binary classification (predicting 2 classes, "spam" and "not spam"), we can replace the output layer as shown below, which will be trainable by default
- Note that we use `BASE_CONFIG["emb_dim"]` (which is equal to 768 in the `"gpt2-small (124M)"` model) to keep the code below more general

In [49]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], 
                                 out_features=num_classes).to(device)

- Technically, it's sufficient to only train the output layer
- However, as I found in [Finetuning Large Language Models](https://magazine.sebastianraschka.com/p/finetuning-large-language-models), experiments show that finetuning additional layers can noticeably improve the performance
- So, we are also making the last transformer block and the final `LayerNorm` module connecting the last transformer block to the output layer trainable

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/10.webp" width=500px>

In [50]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

- We can still use this model similar to before in previous chapters
- For example, let's feed it some text input

In [51]:
device = choose_device(use_accelerator=use_accelerator)

In [52]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).to(device).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape) # shape: (batch_size, num_tokens)

Inputs: tensor([[5211,  345,  423,  640]], device='mps:0')
Inputs dimensions: torch.Size([1, 4])


- What's different compared to previous chapters is that it now has two output dimensions instead of 50,257

In [53]:
with torch.no_grad():
    outputs = model(inputs)

print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens, num_classes)

Outputs:
 tensor([[[ 0.3659,  0.3175],
         [ 0.0619,  0.0078],
         [ 0.0955, -0.5572],
         [ 0.1592,  0.6741]]], device='mps:0')
Outputs dimensions: torch.Size([1, 4, 2])


- As discussed in previous chapters, for each input token, there's one output vector
- Since we fed the model a text sample with 4 input tokens, the output consists of 4 2-dimensional output vectors above

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/11.webp" width=500px>

- In chapter 3, we discussed the attention mechanism, which connects each input token to each other input token
- In chapter 3, we then also introduced the causal attention mask that is used in GPT-like models; this causal mask lets a current token only attend to the current and previous token positions
- Based on this causal attention mechanism, the 4th (last) token contains the most information among all tokens because it's the only token that includes information about all other tokens
- Hence, we are particularly interested in this last token, which we will finetune for the spam classification task

In [54]:
print("Last output token:", outputs[:, -1, :])

Last output token: tensor([[0.1592, 0.6741]], device='mps:0')


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/12.webp" width=200px>

&nbsp;
### 6.6 Calculating the classification loss and accuracy

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/13.webp" width=300px>

- Before explaining the loss calculation, let's have a brief look at how the model outputs are turned into class labels

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/14.webp" width=600px>

In [55]:
print("Last output token:", outputs[:, -1, :])

Last output token: tensor([[0.1592, 0.6741]], device='mps:0')


- Similar to chapter 5, we convert the outputs (logits) into probability scores via the `softmax` function and then obtain the index position of the largest probability value via the `argmax` function

In [56]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("Class label:", label.item())

Class label: 1


- Note that the softmax function is optional here, as explained in chapter 5, because the largest outputs correspond to the largest probability scores

In [57]:
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print("Class label:", label.item())

Class label: 1


- We can apply this concept to calculate the so-called classification accuracy, which computes the percentage of correct predictions in a given dataset
- To calculate the classification accuracy, we can apply the preceding `argmax`-based prediction code to all examples in a dataset and calculate the fraction of correct predictions as follows:

In [58]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

- Let's apply the function to calculate the classification accuracies for the different datasets:

- As we can see, the prediction accuracies are not very good, since we haven't finetuned the model, yet

In [60]:

device = choose_device(use_accelerator=use_accelerator)
print("Device:", device)
model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes
torch.manual_seed(123) # For reproducibility due to the shuffling in the training data loader

train_accuracy = calc_accuracy_loader(mr_train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(mr_val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(mr_test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Device: mps
Training accuracy: 51.00%
Validation accuracy: 0.00%
Test accuracy: 0.00%


- Before we can start finetuning (/training), we first have to define the loss function we want to optimize during training
- The goal is to maximize the spam classification accuracy of the model; however, classification accuracy is not a differentiable function
- Hence, instead, we minimize the cross-entropy loss as a proxy for maximizing the classification accuracy (you can learn more about this topic in lecture 8 of my freely available [Introduction to Deep Learning](https://sebastianraschka.com/blog/2021/dl-course.html#l08-multinomial-logistic-regression--softmax-regression) class)

- The `calc_loss_batch` function is the same here as in chapter 5, except that we are only interested in optimizing the last token `model(input_batch)[:, -1, :]` instead of all tokens `model(input_batch)`

In [4]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Logits of last output token
    #print(logits.shape,target_batch.shape)
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

def exp_calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits.flatten(), target_batch.flatten())
    return loss

The `calc_loss_loader` is exactly the same as in chapter 5

In [5]:
# Same as in chapter 5
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

- Using the `calc_loss_loader`, we compute the initial training, validation, and test set losses before we start training

In [63]:
try:
    from accelerate import Accelerator
    use_accelerator = True
except:
    print("accelerator not available")
    use_accelerator = False


# This worls but gives cpu.  using accelerator we find the Mac's "mps"
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = choose_device()
model.to(device)
with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 3.095
Validation loss: 2.583
Test loss: 2.322


- In the next section, we train the model to improve the loss values and consequently the classification accuracy

&nbsp;
### 6.7 Finetuning the model on supervised data

- In this section, we define and use the training function to improve the classification accuracy of the model
- The `train_classifier_simple` function below is practically the same as the `train_model_simple` function we used for pretraining the model in chapter 5
- The only two differences are that we now 
  1. track the number of training examples seen (`examples_seen`) instead of the number of tokens seen
  2. calculate the accuracy after each epoch instead of printing a sample text after each epoch

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/15.webp" width=500px>

## Training code

Includes some of the functions discussed in the previous section.

In [7]:
# Overall the same as `train_model_simple` in chapter 5
# JMG added save best model feature
from torch import inf

def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter,save_path=None,accuracy_iter=None):
    # Initialize lists to track losses and examples seen
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step,best_accuracy = 0, -1, 0
    if accuracy_iter== None:
        accuracy_iter = 2 * eval_iter
    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            examples_seen += input_batch.shape[0] # New: track examples instead of tokens
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Calculate accuracy after each epoch
        train_accuracy = calc_accuracy_loader(train_loader, model, device, 
                                              num_batches=accuracy_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, 
                                            num_batches=accuracy_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        # A small GPT model takes up 1.4 GB 
        if save_path is not None and val_accuracy > best_accuracy:
            lr = optimizer.param_groups[-1]["lr"]
            save_model (model, optimizer, epoch+1, val_loss, lr, save_path)
            best_accuracy = val_accuracy
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen


def load_model2 (model_file,gpt_config=None):
    if gpt_config is None:
        gpt_config = GPT_CONFIG_124M
    model, optimizer = initialize_model_and_optimizer(gpt_config=gpt_config,
                                                     use_optimizer=True)
    checkpoint = torch.load(model_file, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # return epoch, loss, lr
    return checkpoint['epoch'], checkpoint['loss'], checkpoint['lr']

def load_classifier_model (path):
    """
    Load model downloaded from HuggingFace and saved locally
    in gpt2 dir.  Used on initializing training.
    
    Model size found in global CHOOSE_MODEL.
    """
    #e.g.
    #model_name = "gpt2-small (124M)"
    #CHOOSE_MODEL = model_name
    #BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

    # Initialize base model
    model = GPTModelCLF(BASE_CONFIG)
    # Convert model to classifier as in section 6.5 in ch06.ipynb
    #num_classes = 2
    #model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], 
    #                                 out_features=num_classes)
    # Find path to saved weights
    models_dir ="gpt2"
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    model_dir = os.path.join(models_dir, model_size)
    path = os.path.join(model_dir,model_file)
    # Get a device
    device = choose_device(use_accelerator=use_accelerator)
    # Then load pretrained weights
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    model.to(device)
    model.eval();
    return model
    
def load_classifier_model2 (model_file):
    # We start with all the old weigths of GPT mopdel
    model= make_classifier_model(wt_file_name)
    #model = GPTModel(config)
    #device= choose_device(use_accelerator=use_accelerator)
    #model.to(device)
    # now we load on whatever training has given us.
    checkpoint = torch.load(model_file, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    # return epoch, loss, lr
    return model, checkpoint['epoch'], checkpoint['loss'], checkpoint['lr']

def save_model (model, optimizer,epoch, loss, lr, path):
    torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
            'loss': loss,
            'lr':  lr
            }, path)
    
    

def load_large_model ():
    """
    Loads large GPT model.
    
    Note: no classifier out_head.
    """
    models_dir ="gpt2"
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    model_dir = os.path.join(models_dir, model_size)
    path = os.path.join(model_dir,"model.pt")
    device = choose_device(use_accelerator=use_accelerator)
    model = initialize_GPT_model(path, config=BASE_CONFIG,use_accelerator=use_accelerator)
    #model.load_state_dict(torch.load(path, weights_only=True))
    model.to(device)
    model.eval()
    return model

# Same as chapter 5
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Logits of last output token
    #print(logits.shape,target_batch.shape)
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

# Same as in chapter 5
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


def make_empty_classifier_model(seed=123,num_classes=2,
                         use_accelerator=use_accelerator, 
                         ):
    """
    This is the way Raschke does it in the text and so far this is
    the only way that works for reloading saved weights too.
    
    Using a class defined with the right Linear output (head.out)
    from the getgo doesnt work, and I don;t know why.
    """
    device = choose_device(use_accelerator=use_accelerator)
    torch.manual_seed(seed)
    model = initialize_GPT_model_no_wts(config=BASE_CONFIG,
                                    use_accelerator=use_accelerator)
    model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], 
                                     out_features=num_classes).to(device)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.trf_blocks[-1].parameters():
        param.requires_grad = True
    for param in model.final_norm.parameters():
        param.requires_grad = True
    return model


def load_states(model, model_file="model.pt"):
    # Get a device
    device = choose_device(use_accelerator=use_accelerator)
    # Then load pretrained weights
    params = torch.load(model_file, map_location=device, weights_only=True)
    model.load_state_dict(params["model_state_dict"])
    model.to(device)
    model.eval();

def load_saved_classifier_model (model_file="model.pt",num_classes=2):
    model = make_empty_classifier_model(num_classes=num_classes)
    load_states(model, model_file=model_file)
    return model

def make_classifier_model(path, seed=123,num_classes=2, use_accelerator=use_accelerator, 
                          models_dir="gpt2"
                          ):
    """
    This one initializes with wts, which is unnecessary on reloading
    saved weights.
    """
    if path is None:
        path = get_model_file_name(models_dir=models_dir)
    device = choose_device(use_accelerator=use_accelerator)
    torch.manual_seed(123)
    model = initialize_GPT_model(path, 
                                config=BASE_CONFIG,
                                use_accelerator=use_accelerator)
    model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], 
                                     out_features=num_classes).to(device)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.trf_blocks[-1].parameters():
        param.requires_grad = True
    for param in model.final_norm.parameters():
        param.requires_grad = True
    return model

def get_model_file_name(models_dir="gpt2"):
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    #model_size='774M'
    model_dir = os.path.join(models_dir, model_size)
    return os.path.join(model_dir,"model.pt")
    
###############################################################
####   Retired (with misgivings)
###############################################################

def make_empty_classifier_model2(seed=123,num_classes=2,
                         use_accelerator=use_accelerator, 
                         ):
    """
    This version doesnt work.  No idea why.
    
    Something wrong with the class definition?
    """
    #device = choose_device(use_accelerator=use_accelerator)
    torch.manual_seed(seed)
    #initialize_GPTCLF_model_no_wts
    model = GPTModelCLF(BASE_CONFIG)
    return model



&nbsp;
### 6.8 Using the LLM as a classifier

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/18.webp" width=500px>

- Finally, let's use the finetuned GPT model in action
- The `classify_review` function below implements the data preprocessing steps similar to the `SpamDataset` we implemented earlier
- Then, the function returns the predicted integer class label from the model and returns the corresponding class name

In [42]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # Prepare inputs to the model
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    # Note: In the book, this was originally written as pos_emb.weight.shape[1] by mistake
    # It didn't break the code but would have caused unnecessary truncation (to 768 instead of 1024)

    # Truncate sequences if they too long
    input_ids = input_ids[:min(max_length, supported_context_length)]
    assert max_length is not None, (
        "max_length must be specified. If you want to use the full model context, "
        "pass max_length=model.pos_emb.weight.shape[0]."
    )
    assert max_length <= supported_context_length, (
        f"max_length ({max_length}) exceeds model's supported context length ({supported_context_length})."
    )    
    # Alternatively, a more robust version is the following one, which handles the max_length=None case better
    # max_len = min(max_length,supported_context_length) if max_length else supported_context_length
    # input_ids = input_ids[:max_len]
    
    # Pad sequences to the longest sequence
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # add batch dimension

    # Model inference
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Logits of the last output token
    predicted_label = torch.argmax(logits, dim=-1).item()

    # Return the classified result
    return "spam" if predicted_label == 1 else "not spam"

&nbsp;
## Summary and takeaways

- See the [./gpt_class_finetune.py](./gpt_class_finetune.py) script, a self-contained script for classification finetuning
- You can find the exercise solutions in [./exercise-solutions.ipynb](./exercise-solutions.ipynb)
- In addition, interested readers can find an introduction to parameter-efficient training with low-rank adaptation (LoRA) in [appendix E](../../appendix-E)